# 03 · 扩散与连通响应
扩散只研究最大无向连通分量，输出返回概率和归一化谱熵。小分量使用完整谱；较大分量使用精确零模、细网格SLQ和较少探针/迭代的共同随机数比较。两个领域的最终谱图采用共同的通过收敛阈值的尺度前缀。

In [ ]:
from pathlib import Path
import os, sys
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "prelib").is_dir() and (p / "config.json").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在包含 prelib 和 config.json 的项目目录中运行。")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
for name in ["OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(name, "4")
from threadpoolctl import threadpool_limits
thread_limiter = threadpool_limits(limits=4)
import pandas as pd
from IPython.display import display, Image
from prelib.common import (load_config, domain_config, run_dir, shared_dir,
                           initialize_run, set_active_config, write_json, environment_report)

cfg = load_config()


In [ ]:
from prelib.diffusion import run_spectral
for spec in cfg["domains"]:
    print(spec["label"])
    display(run_spectral(domain_config(cfg, spec["id"])).iloc[::10])


节点删除比较随机与初始度排序；边插入比较随机、短距离优先、长距离优先。展示相对完整图的最大连通分量，表中同时保留全体节点分母、相对曲线面积和半规模比例。目标排序采用初始静态度数。

In [ ]:
from prelib.percolation import run_percolation
for spec in cfg["domains"]:
    dc = domain_config(cfg, spec["id"])
    run_percolation(dc)
    display(pd.read_csv(run_dir(dc) / "tables/connectivity_summary.csv").groupby(["family", "strategy"])[["half_fraction", "relative_AUC"]].mean())
